# Exchange audit: calibration

Reads the calibration phase `build_audit.py` writes to `logs/audit/calibration/<topic>_<model>/calibration_<sub>/`
([`../schedule.md`](../schedule.md) § Exchange audit, Phases): one meeting per stratum, one call per
constructed window. It checks every code path and measures the time per unit, which sizes the audit
jobs. One call per key estimates nothing; § 4 is a first look, never pooled with the audit
([`audit_analysis.ipynb`](audit_analysis.ipynb)). `AUDIT_LOGS` points it at another tree, e.g. a stub run's.

## 1. Load

In [4]:
import os
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)  # every table at three decimals

CWD = Path.cwd().resolve()
REPO = next(p for p in [CWD, *CWD.parents] if (p / "simulation" / "config.py").is_file())
sys.path[:0] = [str(REPO / "analysis"), str(REPO / "analysis" / "audit"), str(REPO / "simulation")]
import build_audit  # noqa: E402
import metrics  # noqa: E402

LOGS = Path(os.getenv("AUDIT_LOGS", REPO / "analysis" / "logs" / "audit" / "calibration"))
TOPICS = ("nuclear", "renewables", "coffee_tea")
MODELS = ("qwen3.5-nodraft:4b", "gemma4-nodraft:e4b", "granite4.2-nodraft:3b")
SUBS = tuple(build_audit.SUBS)

cells = {f"{t}_{m.replace(':', '-')}": LOGS / f"{t}_{m.replace(':', '-')}"
         for t in TOPICS for m in MODELS}
cells = {c: d for c, d in cells.items() if any(d.glob("calibration_*"))}
units = {c: {s: metrics.audit_units(d, s, phases=("calibration",)) for s in SUBS}
         for c, d in cells.items()}


def model(c):
    return c.split("_")[-1].split("-")[0]


print(f"{len(cells)} of {len(TOPICS) * len(MODELS)} cells under {LOGS}")
display(pd.DataFrame({c: {k: metrics.read_params(d / f"calibration_{s}").get(k)
                          for s in SUBS[:1] for k in ("git_commit", "node", "ollama_version", "model_digest")}
                      for c, d in cells.items()}).T.map(lambda v: str(v)[:12]))

9 of 9 cells under C:\Users\jonas\OneDrive\Dokumente\Studium\Inhaltliches\Master\Thesis\thesis\analysis\logs\audit\calibration


,git_commit,node,ollama_version,model_digest
nuclear_qwen3.5-nodraft-4b,939ce13145b5,scc214,0.40.13,b8dd8599dd1f
nuclear_gemma4-nodraft-e4b,939ce13145b5,scc214,0.40.13,a4c02199f27e
nuclear_granite4.2-nodraft-3b,939ce13145b5,scc214,0.40.13,eb4eaff4c3ab
renewables_qwen3.5-nodraft-4b,939ce13145b5,scc214,0.40.13,b8dd8599dd1f
renewables_gemma4-nodraft-e4b,939ce13145b5,scc214,0.40.13,a4c02199f27e
renewables_granite4.2-nodraft-3b,939ce13145b5,scc214,0.40.13,eb4eaff4c3ab
coffee_tea_qwen3.5-nodraft-4b,939ce13145b5,scc214,0.40.13,b8dd8599dd1f
coffee_tea_gemma4-nodraft-e4b,939ce13145b5,scc214,0.40.13,a4c02199f27e
coffee_tea_granite4.2-nodraft-3b,939ce13145b5,scc214,0.40.13,eb4eaff4c3ab


## 2. Code paths

Every key of every kind run exactly once; parse retries, aborts and dropped units; every meeting turn
self-read. Keys and kinds are `build_audit`'s; a sub-phase not yet run is left out.

In [5]:
rows = []
for c, d in cells.items():
    for s, kinds in build_audit.SUBS.items():
        if not (d / f"calibration_{s}" / "events.jsonl").is_file():
            continue   # not run yet
        us = units[c][s]
        keys = Counter(u["key"] for u in us)
        events = Counter(ev["type"] for ev in metrics.read_events(d / f"calibration_{s}")
                         if ev["type"] not in ("start", "unit"))
        calls = [call for u in us for _, call in metrics.position_calls(u)]
        expected = [k for kind in kinds
                    for k in (build_audit.STRATA if kind == "meeting" else build_audit.window_keys(kind))]
        rows.append({"cell": c, "sub": s, "units": len(us),
                     "keys off": sum(keys[k] != 1 for k in expected) + len(set(keys) - set(expected)),
                     "calls": len(calls), "retries": sum(call.get("retries", 0) for call in calls),
                     "no self-read": sum(side["turn"].get("self_read") is None
                                         for u in us if u["kind"] == "meeting"
                                         for side in u["exchange"]["sides"].values()),
                     **events})
pd.DataFrame(rows).fillna(0).set_index(["cell", "sub"])

,,units,keys off,calls,retries,no self-read
cell,sub,,,,,
nuclear_qwen3.5-nodraft-4b,meetings,4,0,40,0,0
nuclear_gemma4-nodraft-e4b,meetings,4,0,40,0,0
nuclear_granite4.2-nodraft-3b,meetings,4,0,40,0,0
renewables_qwen3.5-nodraft-4b,meetings,4,0,40,0,0
renewables_gemma4-nodraft-e4b,meetings,4,0,40,0,0
renewables_granite4.2-nodraft-3b,meetings,4,0,40,0,0
coffee_tea_qwen3.5-nodraft-4b,meetings,4,0,40,0,0
coffee_tea_gemma4-nodraft-e4b,meetings,4,0,40,0,0
coffee_tea_granite4.2-nodraft-3b,meetings,4,0,40,0,0


## 3. Time per unit, projected audit jobs

Mean seconds per unit by kind, and each audit job's hours: that mean times `build_audit.TARGETS` times
the kind's keys, summed over the sub-phase's kinds. Each job's first unit, which loads the model, is left out, and so is
server start; the ceiling is `run_audit.sh`'s `h_rt`.

In [6]:
sec = pd.DataFrame([{"cell": c, "model": model(c), "sub": s, "kind": u["kind"], "seconds": u["seconds"]}
                    for c in cells for s in SUBS for u in units[c][s][1:]])  # [0] loads the model
display(sec.groupby(["model", "kind"]).seconds.describe()[["count", "mean", "50%", "max"]])

n_keys = {kind: len(build_audit.STRATA) if kind == "meeting" else len(build_audit.window_keys(kind))
          for kinds in build_audit.SUBS.values() for kind in kinds}
mean = sec.groupby(["cell", "sub", "kind"]).seconds.mean().reset_index()
mean["hours"] = mean.seconds * mean.kind.map(n_keys) * mean.kind.map(build_audit.TARGETS) / 3600
mean.pivot_table("hours", "cell", "sub", aggfunc="sum")

,,count,mean,50%,max
model,kind,,,,
gemma4,meeting,9.0,2.222,2.22,2.40
granite4.2,meeting,9.0,1.291,1.32,1.40
qwen3.5,meeting,9.0,2.337,2.37,2.48


sub,meetings
cell,
coffee_tea_gemma4-nodraft-e4b,0.483
coffee_tea_granite4.2-nodraft-3b,0.265
coffee_tea_qwen3.5-nodraft-4b,0.489
nuclear_gemma4-nodraft-e4b,0.499
nuclear_granite4.2-nodraft-3b,0.302
nuclear_qwen3.5-nodraft-4b,0.536
renewables_gemma4-nodraft-e4b,0.500
renewables_granite4.2-nodraft-3b,0.293
renewables_qwen3.5-nodraft-4b,0.533


## 4. A first look

Not an estimate: one call per key, pooled here over the topics. The poles each kind of call takes; the
self-read ŝ by argued position s; the probe ô by the partner's argued position and by the partner's
self-read of the same text ([`../schedule.md`](../schedule.md) § Exchange audit, Reported per cell).

In [7]:
def call_kind(path):
    kind = path.rsplit("/", 1)[-1]
    return kind if kind in ("probe", "self_read") else "empty" if kind == "empty" else "declaration"


calls = pd.DataFrame([{"model": model(c), "kind": call_kind(path), "x": call["x"]}
                      for c in cells for s in SUBS for u in units[c][s]
                      for path, call in metrics.position_calls(u)])
display(pd.crosstab([calls.model, calls.kind], calls.x, normalize="index"))

turns = pd.concat([metrics.turns(units[c]["meetings"]).assign(model=model(c)) for c in cells], ignore_index=True)
display(pd.crosstab([turns.model, turns.s], turns.self_read, normalize="index"))

perc = pd.concat([metrics.perception(units[c]["meetings"]).assign(model=model(c)) for c in cells], ignore_index=True)
display(pd.crosstab([perc.model, perc.partner_s], perc.o_hat, normalize="index"))
display(pd.crosstab([perc.model, perc.partner_self_read], perc.o_hat, normalize="index"))

x                          -1      1
model      kind                     
gemma4     declaration  0.500  0.500
           empty        0.167  0.833
           probe        0.500  0.500
           self_read    0.500  0.500
granite4.2 declaration  0.438  0.562
           empty        0.333  0.667
           probe        0.500  0.500
           self_read    0.500  0.500
qwen3.5    declaration  0.479  0.521
           empty        0.208  0.792
           probe        0.500  0.500
           self_read    0.500  0.500

self_read       -1    1
model      s           
gemma4     -1  1.0  0.0
            1  0.0  1.0
granite4.2 -1  1.0  0.0
            1  0.0  1.0
qwen3.5    -1  1.0  0.0
            1  0.0  1.0

o_hat                  -1    1
model      partner_s          
gemma4     -1         1.0  0.0
            1         0.0  1.0
granite4.2 -1         1.0  0.0
            1         0.0  1.0
qwen3.5    -1         1.0  0.0
            1         0.0  1.0

o_hat                          -1    1
model      partner_self_read          
gemma4     -1                 1.0  0.0
            1                 0.0  1.0
granite4.2 -1                 1.0  0.0
            1                 0.0  1.0
qwen3.5    -1                 1.0  0.0
            1                 0.0  1.0

## 5. Meetings: empty against one-entry declarations

Each meeting agent declares once from empty memory, the declared prior both conditions share, and
once per condition from the exchange's one entry: `full` (both turns), `own` (the own turn alone).
Per model and topic the mean declaration and the self-read ŝ and probe ô; per model the share
declaring exactly ŝ; pooled, the mean by ŝ and, per condition, by ŝ and ô. With one entry, habit
moves y with ŝ and feedback with ô ([README](../README.md#signatures) § Signatures).

In [8]:
CONDS = ("empty", "full", "own")
decl = pd.DataFrame([{"topic": c.rsplit("_", 1)[0], "model": model(c),
                      "s_hat": side["turn"]["self_read"]["x"], "o_hat": side["probe"]["x"],
                      **{k: d[k]["x"] for k in CONDS}}
                     for c in cells for u in units[c]["meetings"]
                     for role, d in u["declarations"].items()
                     for side in [u["exchange"]["sides"][role]]])
display(decl.groupby(["topic", "model"])[[*CONDS, "s_hat", "o_hat"]].mean())
display(pd.DataFrame({k: decl[k].eq(decl.s_hat).groupby(decl.model).mean() for k in CONDS}))
display(decl.groupby("s_hat")[list(CONDS)].mean().assign(n=decl.groupby("s_hat").size()))
for k in ("full", "own"):
    print(k)
    display(decl.pivot_table(k, "s_hat", "o_hat", aggfunc="mean"))
display(decl.pivot_table("full", "s_hat", "o_hat", aggfunc="size"))

empty  full   own  s_hat  o_hat
topic      model                                      
coffee_tea gemma4       0.75  0.00  0.00    0.0    0.0
           granite4.2  -0.75  0.25  0.00    0.0    0.0
           qwen3.5      0.50  0.00 -0.25    0.0    0.0
nuclear    gemma4       0.25  0.00  0.00    0.0    0.0
           granite4.2   0.75  0.25  0.00    0.0    0.0
           qwen3.5      0.50  0.50  0.25    0.0    0.0
renewables gemma4       1.00  0.00  0.00    0.0    0.0
           granite4.2   1.00  0.25  0.00    0.0    0.0
           qwen3.5      0.75 -0.25  0.00    0.0    0.0

,empty,full,own
model,,,
gemma4,0.500,1.000,1.000
granite4.2,0.417,0.875,1.000
qwen3.5,0.375,0.875,0.917


,empty,full,own,n
s_hat,,,,
-1,0.667,-0.722,-0.944,36
1,0.389,0.944,0.944,36


full


o_hat,-1,1
s_hat,,
-1,-0.778,-0.667
1,1.000,0.889


own


o_hat,-1,1
s_hat,,
-1,-0.889,-1.0
1,0.889,1.0


o_hat,-1,1
s_hat,,
-1,18,18
1,18,18
